In [ ]:
from sklearn.datasets import make_blobs # used for data
import matplotlib.pyplot as plt
import math

In [ ]:
X, Y = make_blobs(cluster_std=2)
train_x, train_y, test_x, test_y = X[5:], Y[5:], X[:5], Y[:5]

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(train_x[:,0], train_x[:,1], c=train_y)
plt.scatter(test_x[:,0], test_x[:,1], c='red', marker='X', s=80);

## Basic

In [ ]:
# Application of Bayes Rule where P(C|x) = (P(C) * P(x|C)) / P(x)
## but P(C) * P(x|C) = P(x,C) = P(x0,x1,...,C)

# Now assuming independence we can get to P(C|x) = (P(C) * P(x0|C) * P(x1|C) ...) / p(x)
## and p(x) = P(C0|x)P(C0)+..., which can be removed as constant scaling factor

# Finally: P(C|x) = P(C) * P(x0|C) * P(x1|C) ...

class NaiveBayes:
    """
    For a given class index C, we can fit and use:
    P(C|x) = P(C) * P(x0|C) * P(x1|C) ... * P(xN|C)
    """
    
    def __init__(self, distribution="gauss"):
        self.obs_by_class = {}
        self.priors = {}
        self.feature_dist_args = {}
        if distribution != "gauss":
            raise ValueError("Only Gaussian NaiveBayes model supported for now")
        
    def fit(self, x, y):
        # Get priors
        self.priors = {}
        for i,target in enumerate(y):
            if target not in self.priors:
                self.priors[target] = 0
            self.priors[target] +=1    
            
            if target not in self.obs_by_class:
                self.obs_by_class[target] = []
            self.obs_by_class[target].append(x[i])
        
        self.priors = {c: p/y.shape[0] for c,p in self.priors.items()}
        
        # Fit per feature gaussian
        for _class, cls_samples in self.obs_by_class.items():
            self.feature_dist_args[_class] = []
            for f_index in range(cls_samples[0].shape[0]):
                feature_vals = [sample[f_index] for sample in cls_samples]
                mean = sum(feature_vals) / len(cls_samples)
                var = sum([(f-mean)**2 for f in feature_vals]) / len(cls_samples)
                self.feature_dist_args[_class].append([mean,var])

    def predict(self, x):
        """P(C|x) = P(C) * P(x0|C) * P(x1|C) ... * P(xN|C)"""
        probs_by_class = {}
        for _class, _prior_prob in self.priors.items():
            probs_by_class[_class] = _prior_prob
            for f_index, dist_args in enumerate(self.feature_dist_args[_class]):
                feature_prob = self.gauss_pdf(x[f_index], *dist_args)
                probs_by_class[_class] *= feature_prob
        return [int(c) for c, p in probs_by_class.items() if p==max(probs_by_class.values())][0]
        
    def gauss_pdf(self, xi, mean, var):
        return (1 / math.sqrt(2 * math.pi * var)) * math.exp(-((xi - mean) ** 2) / (2 * var))


In [ ]:
model = NaiveBayes()
model.fit(train_x, train_y)
assignments = [model.predict(x) for x in test_x]

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(train_x[:,0], train_x[:,1], c=train_y)
plt.scatter(test_x[:,0], test_x[:,1], c=assignments, marker='X', s=80);

## Advanced (cache, log domain, online update)

In [ ]:
class NaiveBayes:
    """
    Gaussian Naive Bayes with:
      1. Cached Gaussian constants (data-independent terms)
      2. Log-domain probability accumulation
      3. Incremental/online updating of mean & variance
    """

    def __init__(self, distribution="gauss"):
        self.obs_by_class = {}
        self.priors = {}
        self.feature_dist_args = {}
        self.cached_log_consts = {}   # (1) cache for Gaussian constants
        self.class_counts = {}
        if distribution != "gauss":
            raise ValueError("Only Gaussian NaiveBayes model supported for now")

    def fit(self, x, y):
        """Batch training."""
        self.obs_by_class = {}
        self.class_counts = {}

        # Collect samples per class
        for i, target in enumerate(y):
            if target not in self.obs_by_class:
                self.obs_by_class[target] = []
            self.obs_by_class[target].append(x[i])
            self.class_counts[target] = self.class_counts.get(target, 0) + 1

        total = len(y)
        self.priors = {c: n / total for c, n in self.class_counts.items()}

        # Compute mean, var, and cache Gaussian constants
        for c, samples in self.obs_by_class.items():
            num_features = len(samples[0])
            self.feature_dist_args[c] = []
            self.cached_log_consts[c] = []

            for f_index in range(num_features):
                feature_vals = [s[f_index] for s in samples]
                mean = sum(feature_vals) / len(feature_vals)
                var = sum((f - mean) ** 2 for f in feature_vals) / len(feature_vals)
                var = max(var, 1e-6)  # avoid divide-by-zero
                self.feature_dist_args[c].append([mean, var])

                # (1) Cache log constant term: -0.5 * log(2πσ²)
                log_const = -0.5 * math.log(2 * math.pi * var)
                self.cached_log_consts[c].append(log_const)

    def predict(self, x):
        """
        (2) Work in log domain:
            log P(C|x) ∝ log P(C) + Σ log P(x_i|C)
        """
        log_probs = {}
        for c, prior in self.priors.items():
            log_prob = math.log(prior)

            for f_index, (mean, var) in enumerate(self.feature_dist_args[c]):
                log_const = self.cached_log_consts[c][f_index]

                # log of Gaussian PDF (no recomputed constants)
                log_likelihood = log_const - ((x[f_index] - mean) ** 2) / (2 * var)
                log_prob += log_likelihood

            log_probs[c] = log_prob

        # choose class with highest log-probability
        return max(log_probs, key=log_probs.get)

    def update(self, x, y):
        """
        (3) Incremental / online update of mean, var, priors, and cached constants.
        """
        if y not in self.feature_dist_args:
            # First sample of a new class
            self.feature_dist_args[y] = [[xi, 1e-6] for xi in x]
            self.cached_log_consts[y] = [-0.5 * math.log(2 * math.pi * 1e-6)] * len(x)
            self.class_counts[y] = 1
        else:
            n = self.class_counts[y]
            self.class_counts[y] += 1

            # Incrementally update each feature’s mean/variance
            for f_index, (mean, var) in enumerate(self.feature_dist_args[y]):
                new_mean = mean + (x[f_index] - mean) / self.class_counts[y]
                new_var = var + (x[f_index] - mean) * (x[f_index] - new_mean)
                new_var = max(new_var, 1e-6)

                self.feature_dist_args[y][f_index] = [new_mean, new_var]

                # (1) Re-cache constant for this feature
                self.cached_log_consts[y][f_index] = -0.5 * math.log(2 * math.pi * new_var)

        # Update priors
        total = sum(self.class_counts.values())
        self.priors = {c: n / total for c, n in self.class_counts.items()}


In [ ]:
model = NaiveBayes()
model.fit(train_x, train_y)
assignments = [model.predict(x) for x in test_x]

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(train_x[:,0], train_x[:,1], c=train_y)
plt.scatter(test_x[:,0], test_x[:,1], c=assignments, marker='X', s=80);